# Stage 2 — FULL projection + analyze + probes (Qwen3-32B)

**Paper run** — all normalized trajs (no pilot subsample). Thinking ON. Primary layer **49**.

**Runtime:** Colab **A100 80GB** (bf16 32B). One GPU — do not set `CUDA_VISIBLE_DEVICES`.

**Outputs (on Drive):**
- `projections_full.parquet`
- `agentic_meanpool_32b_full.npz` (Stage-5 probes)
- `analysis_report_full/`, `probe_report_full/`

**Requirements**
1. `value_axis_32b.npy` + `axis_manifest_32b.json`
2. Zip of **all** normalized traj JSONs (both outcomes)
3. Drive mounted — checkpoints survive disconnects

**ETA:** often **~10–15 h** wall-clock (~200 trajs). Resume is built in — re-run the project cell.

**Disk:** activations rewrite needs free space (~several GB). Drive is usually fine; watch Colab disk if you copy the npz locally.

Fresh run for probes: use empty `stage2_full_32b` dirs (or delete old broken Hub `.npz`). Do **not** mix with `stage2_pilot_32b`.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Need GPU (A100 80GB for Qwen3-32B bf16)'
name = torch.cuda.get_device_name(0)
vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print(name, 'VRAM GB:', vram)
assert vram >= 70, (
    f'Got {vram} GB — need A100 80GB / high-RAM. '
    'Runtime → Change runtime type → A100.'
)

In [ ]:
import os
REPO = '/content/failure_prediction_research'
if not os.path.isdir(REPO):
    !git clone https://github.com/abdelmagid07/failure_prediction_research.git {REPO}
else:
    %cd {REPO}
    !git pull
%cd {REPO}
!pip install -q -e stage1 -e stage2
!pip install -q pyarrow pandas scikit-learn matplotlib

In [ ]:
# Drive for outputs (survives disconnects). Keep growing npz HERE, not only in /content.
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/failure_prediction_research/stage2_full_32b')
OUT_DIR = DRIVE_ROOT / 'outputs'
ACT_DIR = DRIVE_ROOT / 'activations'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ACT_DIR.mkdir(parents=True, exist_ok=True)

print('DRIVE_ROOT:', DRIVE_ROOT)
print('OUT_DIR:', OUT_DIR)
print('ACT_DIR:', ACT_DIR)

# If resuming a failed Hub run: delete a broken npz before project, keep parquet only if you
# accept axis-only resume without rebuilding activations for those trajs.
# For a clean probe rebuild, remove BOTH parquet and npz under this Drive folder.

In [ ]:
PRIMARY_LAYER = 49
MODEL = 'Qwen/Qwen3-32B'
N_LAYERS = 64
SAVE_ACTIVATIONS = True  # required for Stage-5 probes

print('PRIMARY_LAYER', PRIMARY_LAYER)
print('MODEL', MODEL)
print('SAVE_ACTIVATIONS', SAVE_ACTIVATIONS)

## Upload inputs (three dialogs)

1. `value_axis_32b.npy`
2. `axis_manifest_32b.json`
3. Zip of **all** normalized trajectory JSONs

In [ ]:
import json, shutil, zipfile
from pathlib import Path
from google.colab import files
import numpy as np

REPO = Path('/content/failure_prediction_research')
AXIS_DIR = REPO / 'stage1' / 'data'
NORM_DIR = REPO / 'stage2' / 'data' / 'normalized_full'
AXIS_DIR.mkdir(parents=True, exist_ok=True)
NORM_DIR.mkdir(parents=True, exist_ok=True)

def _take_one(upload_dict, dest: Path, *, expect_suffix: str | None = None):
    assert len(upload_dict) == 1, f'Upload exactly one file, got {list(upload_dict)}'
    name = next(iter(upload_dict))
    src = Path(name)
    if expect_suffix is not None:
        assert src.suffix.lower() == expect_suffix.lower(), (
            f'Expected {expect_suffix}, got {src.name}'
        )
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        dest.unlink()
    shutil.move(str(src), dest)
    print('saved ->', dest)
    return dest

print('1/3 — upload value_axis_32b.npy')
AXIS = _take_one(files.upload(), AXIS_DIR / 'value_axis_32b.npy', expect_suffix='.npy')
axis = np.load(AXIS)
print('axis shape', axis.shape, '(expect 64 x 5120)')
assert axis.shape == (64, 5120), axis.shape

print('2/3 — upload axis_manifest_32b.json')
MANIFEST = _take_one(
    files.upload(), AXIS_DIR / 'axis_manifest_32b.json', expect_suffix='.json'
)
manifest = json.loads(MANIFEST.read_text())
print(
    'manifest primary_layer=',
    manifest.get('primary_layer'),
    'primary_auroc=',
    manifest.get('primary_auroc'),
    'enable_thinking=',
    manifest.get('enable_thinking'),
)

print('3/3 — upload normalized trajectories zip')
z_up = files.upload()
assert len(z_up) == 1, f'Upload exactly one zip, got {list(z_up)}'
zname = next(iter(z_up))
assert Path(zname).suffix.lower() == '.zip', zname

extract = REPO / 'stage2' / 'data' / '_full_zip_extract'
if extract.exists():
    shutil.rmtree(extract)
extract.mkdir(parents=True)
with zipfile.ZipFile(zname) as z:
    z.extractall(extract)

for p in NORM_DIR.glob('*.json'):
    p.unlink()
n = 0
skipped = 0
for p in extract.rglob('*.json'):
    if p.name == 'ingest_manifest.json':
        skipped += 1
        continue
    d = json.loads(p.read_text())
    if not isinstance(d, dict) or 'steps' not in d or 'outcome' not in d:
        skipped += 1
        continue
    shutil.copy2(p, NORM_DIR / p.name)
    n += 1
print(f'normalized trajs: {n} in {NORM_DIR}'
      + (f' (skipped {skipped} non-traj)' if skipped else ''))
assert n >= 150, f'Expected full set (~198); got {n}'

In [ ]:
# Outcome sanity — full set, both classes, no subsample
import json
from collections import Counter
from pathlib import Path

NORM_DIR = Path('/content/failure_prediction_research/stage2/data/normalized_full')
outs, steps = [], []
for p in NORM_DIR.glob('*.json'):
    d = json.loads(p.read_text())
    outs.append(int(d['outcome']))
    steps.append(d.get('n_steps') or len(d.get('steps', [])))
print('n', len(outs), 'outcomes', Counter(outs))
print('steps mean/median/max',
      round(sum(steps)/len(steps), 1), sorted(steps)[len(steps)//2], max(steps))
assert 0 in outs and 1 in outs

In [ ]:
# Optional: wipe Drive checkpoints for a clean probe rebuild (uncomment to run)
# from pathlib import Path
# for p in [
#     OUT_DIR / 'projections_full.parquet',
#     ACT_DIR / 'agentic_meanpool_32b_full.npz',
# ]:
#     if p.exists():
#         p.unlink()
#         print('removed', p)

## Project (GPU, long)

All layers + `--activations-npz`. Checkpoints after **each traj** on Drive. Re-run to **resume**.

Keep the tab awake; disconnecting the runtime stops the job (Drive keeps finished trajs).

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path('/content/failure_prediction_research')
NORM_DIR = REPO / 'stage2' / 'data' / 'normalized_full'
AXIS = REPO / 'stage1' / 'data' / 'value_axis_32b.npy'
PROJ = OUT_DIR / 'projections_full.parquet'
ACT_NPZ = ACT_DIR / 'agentic_meanpool_32b_full.npz'

cmd = [
    sys.executable, '-u', '-m', 'stage2.extract.project_steps',
    '--traj-dir', str(NORM_DIR),
    '--axis-path', str(AXIS),
    '--model', MODEL,
    '--n-layers', str(N_LAYERS),
    '--enable-thinking',
    '--output', str(PROJ),
]
if SAVE_ACTIVATIONS:
    cmd.extend(['--activations-npz', str(ACT_NPZ)])

print('CMD:', ' '.join(cmd), flush=True)
print('Quiet while loading Qwen3-32B is normal...', flush=True)

proc = subprocess.Popen(
    cmd, cwd=str(REPO / 'stage2'),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print('project_steps exit:', rc, flush=True)
assert rc == 0 and PROJ.exists(), PROJ
print('projections ->', PROJ, 'size MB', round(PROJ.stat().st_size / 1e6, 2))
if SAVE_ACTIVATIONS:
    print('activations ->', ACT_NPZ, 'exists', ACT_NPZ.exists(),
          'MB', round(ACT_NPZ.stat().st_size / 1e6, 1) if ACT_NPZ.exists() else None)

## Analyze (CPU)

Headline metrics use `PRIMARY_LAYER` (49).

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path('/content/failure_prediction_research')
PROJ = OUT_DIR / 'projections_full.parquet'
REPORT = OUT_DIR / 'analysis_report_full'
REPORT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, '-u', '-m', 'stage2.analyze.run_analyses',
    '--projections', str(PROJ),
    '--output-dir', str(REPORT),
    '--primary-layer', str(PRIMARY_LAYER),
]
print('CMD:', ' '.join(cmd), flush=True)
proc = subprocess.Popen(
    cmd, cwd=str(REPO / 'stage2'),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print('run_analyses exit:', rc, flush=True)
assert rc == 0
print('report ->', REPORT)

## Probes (Stage 5)

Needs a valid `agentic_meanpool_32b_full.npz`. Copy to local `/tmp` for faster fit if Drive is slow.

In [ ]:
import subprocess, sys, shutil
from pathlib import Path

REPO = Path('/content/failure_prediction_research')
ACT_NPZ = ACT_DIR / 'agentic_meanpool_32b_full.npz'
PROBE_DIR = OUT_DIR / 'probe_report_full'

if not ACT_NPZ.exists():
    print('SKIP probes — missing', ACT_NPZ)
else:
    local_act = Path('/tmp/agentic_meanpool_32b_full.npz')
    print('copying npz local for faster probe fit...', flush=True)
    shutil.copy2(ACT_NPZ, local_act)
    cmd = [
        sys.executable, '-u', '-m', 'stage2.probes.fit_probes',
        '--activations', str(local_act),
        '--output-dir', str(PROBE_DIR),
    ]
    print('CMD:', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(
        cmd, cwd=str(REPO / 'stage2'),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
    print('fit_probes exit:', proc.wait(), flush=True)
    print('probe report ->', PROBE_DIR)

In [ ]:
# Preview headline metrics
import json
from pathlib import Path
from IPython.display import Image, display

REPORT = OUT_DIR / 'analysis_report_full'
rep = REPORT / 'analysis_report.json'
if rep.exists():
    print(json.dumps(json.loads(rep.read_text()), indent=2)[:4000])

for name in ['final_step_separation.png', 'noise_by_token_type.png']:
    p = REPORT / name
    if p.exists():
        print(name)
        display(Image(filename=str(p)))
    else:
        print('missing', p)

In [ ]:
# Zip small artifacts (npz stays on Drive)
import zipfile
from pathlib import Path
from google.colab import files

zip_path = OUT_DIR / 'stage2_full_32b_results_small.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob('*'):
        if p.is_file() and p.suffix.lower() in {'.json', '.png', '.csv', '.parquet'}:
            z.write(p, p.relative_to(OUT_DIR).as_posix())
print('zip ->', zip_path)
print('activations on Drive:', ACT_DIR / 'agentic_meanpool_32b_full.npz')
files.download(str(zip_path))